# Baseline Evaluation and Indexing

This notebook establishes the performance floor (Baseline) for the retrieval system. It builds the initial Sparse Indices (SPLADE and BM25) and evaluates the pre-trained BGE-Base model across multiple dimensions. This serves as the control group to measure the lift provided by Matryoshka fine-tuning later.

In [2]:
import sys, os, yaml, shutil, torch, pickle
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Path Setup
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
from src.esci.data import sample_dataset
from src.esci.system_a import encode_systemA
from src.esci.system_b import build_candidates
from src.esci.artifacts import load_artifacts
from src.esci.sparse_retrievers import SPLADEFast, BM25Fast
from src.esci.metrics import compute_recall_metrics, compute_ndcg_metrics


# Load Config
with open(project_root / "configs" / "esci.yaml") as f:
    cfg = yaml.safe_load(f)

# Load Data
print("Loading Processed Data...")
pair_df = pd.read_parquet(cfg["paths"]["processed_dir"] + "/pair_df.parquet")
print("Processed Data is LOADED")

Loading Processed Data...
Processed Data is LOADED


### Ground Truth Construction

In [3]:
# We pre-calculate the 'q_rels' (Query Relevance) map.
# This dictionary maps query_id -> [list of grades].
# It acts as the immutable standard against which all retrieval strategies 
# (Dense, Sparse, Hybrid) will be measured using nDCG and Recall.

print("Building Ground Truth Map (q_rels)...")
q_rels = pair_df.groupby("query_id")["grade"].apply(list).to_dict()
print("DONE")

Building Ground Truth Map (q_rels)...
DONE


### Clear Aartifacts & Encode

To save storage space and avoid confusion between artifacts from baseline model vs finetuned model, we clear the artifacts folder once we start encoding with the baseline model.

In [4]:
artifact_dir = Path(cfg["paths"]["artifacts_dir"])

if os.path.exists(artifact_dir):
    print(f"Clearing old artifacts in {artifact_dir}...")
    shutil.rmtree(artifact_dir)
    os.makedirs(artifact_dir)

base_model = cfg["biencoder_model"]
print(f"Encoding with Baseline Model: {base_model}")
encode_systemA(pair_df, cfg, model_override=base_model)

Clearing old artifacts in ..\artifacts\systemA...
Encoding with Baseline Model: BAAI/bge-base-en-v1.5
 Encoding with System A model: BAAI/bge-base-en-v1.5
 Max Seq Length: 256
    -> Preparing Product List...
    -> Encoding 164001 Products...


Batches:   0%|          | 0/5126 [00:00<?, ?it/s]

    -> Preparing Query List...
    Applying BGE instruction prefix: 'Represent this sentence for searching relevant passages: '
    -> Encoding 8908 Queries...


Batches:   0%|          | 0/279 [00:00<?, ?it/s]

 System A Encoding Complete.


### Build Sparse Indices

In [5]:
print("Pre-building Sparse Indices...")
_, _, prod_df, _, _ = load_artifacts("us", "test", cfg)

# Build SPLADE
splade = SPLADEFast(
    cfg["sparse"]["splade_model"],
    batch_size=cfg["sparse"]["batch_size"]
)

splade.build_index(
    prod_df["product_text_dense"].tolist(),
    prod_df["product_id"].tolist()
)

print("SPLADE built:", splade.doc_matrix.shape)

Pre-building Sparse Indices...


C:\Users\aminl\anaconda3\envs\pytorch-gpu\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


 Building Fast SPLADE index for 164001 docs...
 SPLADE index built. Matrix shape: (164001, 30522)
SPLADE built: torch.Size([164001, 30522])


In [6]:
# Build BM25
bm25 = BM25Fast()
bm25.build_index(
    texts=prod_df["product_title"].fillna("").astype(str).tolist(),
    pids=prod_df["product_id"].tolist()
)

 Building Fast GPU BM25 Index (stemming + normalize) for 164001 docs...
 BM25 index built. Matrix shape: (164001, 96275)
 BM25 vocab size: 96275


### Saving artifacts

In [7]:
save_dir = "../artifacts/systemA"
os.makedirs(save_dir, exist_ok=True)

In [8]:
splade_path = f"{save_dir}/splade_data.pt"
torch.save({
    "doc_matrix": splade.doc_matrix,
    "pid_list": splade.pid_list,
    "model_state_dict": splade.model.state_dict()
}, splade_path)

print(f"SPLADE saved to: {splade_path}")

SPLADE saved to: ../artifacts/systemA/splade_data.pt


In [9]:
bm25_path = f"{save_dir}/bm25_retriever.pkl"
with open(bm25_path, "wb") as f:
    pickle.dump(bm25, f)

print(f"BM25 saved to: {bm25_path}")

BM25 saved to: ../artifacts/systemA/bm25_retriever.pkl


### Loading Artifacts

In [10]:
# Loading SPLADE/BM25 from a .pt checkpoint is ~10x faster than rebuilding it.
# We explicitly map the sparse matrix to the GPU ('cuda') to ensure
# that subsequent scoring operations utilize Tensor Cores.

print("Loading SPLADE Matrix...")
splade_data = torch.load("../artifacts/systemA/splade_data.pt", map_location="cpu")

splade_loaded = SPLADEFast(
    cfg["sparse"]["splade_model"], 
    batch_size=cfg["sparse"]["batch_size"]
)

splade_loaded.doc_matrix = splade_data["doc_matrix"]
splade_loaded.pid_list   = splade_data["pid_list"]

# CRITICAL: device alignment
splade_loaded.doc_matrix = splade_loaded.doc_matrix.to(splade_loaded.device)

# Safety
assert splade_loaded.doc_matrix.shape[0] == len(splade_loaded.pid_list)

print("Loading BM25 Index...")
with open("../artifacts/systemA/bm25_retriever.pkl", "rb") as f:
    bm25_loaded = pickle.load(f)

splade = splade_loaded
bm25 = bm25_loaded

Loading SPLADE Matrix...


C:\Users\aminl\anaconda3\envs\pytorch-gpu\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loading BM25 Index...


## Running Experiment Loop

In [15]:
# Hyperparametes used for the experiment:
# rrf_k: 60
# rrf_weights: "dense": 1.0, "splade": 0.5, "bm25": 0.3

In [11]:
results = []
dims = cfg["matryoshka"]["dims"]
strategies = {
    "Dense Only": ["dense"],
    "Dense + BM25": ["dense", "bm25"],
    "Dense + SPLADE": ["dense", "splade"],
    "Dense + BM25 + SPLADE": ["dense", "bm25", "splade"]
}

max_len = cfg["matryoshka"]["max_seq_length"]

for strat, sources in strategies.items():
    print(f"\n Strategy: {strat}")
    cfg["retrieval"]["sources"] = sources
    
    for dim in dims:
        df, qps = build_candidates(
            cfg, split="test", override_dim=dim,
            prebuilt_splade=splade if "splade" in sources else None,
            prebuilt_bm25=bm25 if "bm25" in sources else None
        )
        
        # Pass q_rels here!
        # Compute them separately
        recall_results = compute_recall_metrics(df, q_rels)
        ndcg_results = compute_ndcg_metrics(df, q_rels)
        
        # Merge them into one final dictionary
        final_metrics = {**recall_results, **ndcg_results}
        
        results.append({
            "Model": "Baseline",
            "Strategy": strat,
            "Dimension": dim,
            "Max Length": max_len,
            "Recall@50": final_metrics["Recall@50"],
            "Recall@100": final_metrics["Recall@100"],
            "Recall@200": final_metrics["Recall@200"],
            "nDCG@10": final_metrics["nDCG@10"],
            "nDCG@20": final_metrics["nDCG@20"],
            "nDCG@50": final_metrics["nDCG@50"],
            "QPS": round(qps, 2)
        })


 Strategy: Dense Only
 Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...

 Strategy: Dense + BM25
 Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...

 Strategy: Dense + SPLADE
 Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=64)
    -> Me

In [12]:
# Save & Display
output_dir = "../results"
if not os.path.exists(output_dir): os.makedirs(output_dir)
results_df = pd.DataFrame(results)
results_df.to_csv(f"{output_dir}/baseline_metrics_per_dim.csv", index=False)

In [13]:
print("\n QPS & Performance by Dimension:")
display(results_df.pivot(index="Dimension", columns="Strategy", values=["Recall@200", "QPS"]))


 QPS & Performance by Dimension:


Recall@200                                                  \
Strategy  Dense + BM25 Dense + BM25 + SPLADE Dense + SPLADE Dense Only   
Dimension                                                                
64            0.557411              0.700460       0.660043   0.426956   
128           0.659875              0.734503       0.710429   0.588258   
256           0.721953              0.760375       0.745396   0.681114   
512           0.752546              0.774440       0.764448   0.724900   
768           0.762853              0.780489       0.772691   0.739768   

                   QPS                                                  
Strategy  Dense + BM25 Dense + BM25 + SPLADE Dense + SPLADE Dense Only  
Dimension                                                               
64              180.56                 80.99          87.15     446.34  
128             111.40                 71.06          77.19     244.61  
256              81.65                 52.23          57.73     131.59  
512              51.92                 36.17          38.49      68.60  
768              36.39                 26.48          27.60      41.90

In [14]:
results_df

,Model,Strategy,Dimension,Max Length,Recall@50,Recall@100,Recall@200,nDCG@10,nDCG@20,nDCG@50,QPS
0,Baseline,Dense Only,768,256,0.569735,0.662507,0.739768,0.433009,0.463072,0.515612,41.90
1,Baseline,Dense Only,512,256,0.555104,0.647921,0.724900,0.422693,0.451474,0.503189,68.60
2,Baseline,Dense Only,256,256,0.511757,0.601916,0.681114,0.395021,0.420819,0.469609,131.59
3,Baseline,Dense Only,128,256,0.430857,0.512219,0.588258,0.338874,0.360226,0.403421,244.61
4,Baseline,Dense Only,64,256,0.290365,0.357380,0.426956,0.235852,0.249192,0.281322,446.34
5,Baseline,Dense + BM25,768,256,0.592636,0.685852,0.762853,0.448713,0.481051,0.534117,36.39
6,Baseline,Dense + BM25,512,256,0.580858,0.674523,0.752546,0.441316,0.472488,0.524907,51.92
7,Baseline,Dense + BM25,256,256,0.544849,0.639521,0.721953,0.423910,0.451498,0.500605,81.65
8,Baseline,Dense + BM25,128,256,0.474590,0.566390,0.659875,0.386450,0.406928,0.450027,111.40
9,Baseline,Dense + BM25,64,256,0.344066,0.431098,0.557411,0.306621,0.315276,0.348642,180.56
